# Module 04: Building an MCP Tool Server with FastMCP

## Learning Objectives
By completing this notebook, you will:
1. Build a SQLite company database and verify it with direct queries
2. Implement a three-tool MCP server using FastMCP decorators
3. Introspect tool schemas the same way an RL agent runtime (ART) does at startup
4. Trace a complete agent trajectory — from question to JOIN query — step by step
5. Design and implement your own MCP server for a new domain

## Prerequisites
- Module 02: ART Framework (how the agent interacts with tool servers)
- Module 03: RULER Rewards (why structured tool use improves reward signal)
- Basic SQL: SELECT, JOIN, WHERE, GROUP BY

## Estimated Time: 15 minutes

---

**Key Insight:** The MCP server is the agent's entire world during training. Three tools — `list_tables`, `describe_table`, `run_query` — force the agent to learn the correct schema-first exploration pattern before writing any SQL. The server is deliberately minimal: every additional tool dilutes that signal.

## 1. Setup and Imports

All dependencies are standard library (`sqlite3`, `json`, `pathlib`) plus `fastmcp`, which was installed in the course environment. We also import `asyncio` because FastMCP's tool-calling and introspection APIs are async — they are designed to run inside an async agent loop.

`nest_asyncio` patches Python's event loop so that `asyncio.run()` works inside Jupyter, which already runs its own event loop. This is a notebook-only workaround — production code using FastMCP runs in a normal async context and does not need it.

In [ ]:
# Purpose: Import all dependencies for this notebook
# Key Concept: fastmcp wraps decorated Python functions into MCP-compliant tools;
#              sqlite3 is the database backend; asyncio drives the tool-call loop.

import sqlite3
import json
import asyncio
from pathlib import Path
from typing import Any

import fastmcp

# nest_asyncio allows asyncio.run() to work inside Jupyter's existing event loop.
# Required in notebooks only — remove this in production scripts.
import nest_asyncio
nest_asyncio.apply()

# Reproducibility: pin random seed for any stochastic operations later
import random
random.seed(42)

# Database lives next to this notebook for easy inspection
DB_PATH = Path("company.db")

print(f"fastmcp version : {fastmcp.__version__}")
print(f"Database path   : {DB_PATH.resolve()}")
print("Setup complete.")

## 2. Create the Company Database

### Why this schema?

Three tables with realistic JOIN paths give the agent enough variety to generalise:

| Table | What it models | Key relationships |
|---|---|---|
| `departments` | Organisational units with budgets | `manager_id` → `employees.id` |
| `employees` | Staff with salaries and roles | `dept_id` → `departments.id` |
| `projects` | Active and historical work | `lead_id` → `employees.id`, `dept_id` → `departments.id` |

The `projects.status` column (`'active'`, `'completed'`, `'cancelled'`) creates natural filter conditions. The `employees.is_active` flag adds another dimension for WHERE clauses.

**Modification point:** The data below is the canonical training set. In the exercise you will create an entirely different schema — a bookstore — using exactly this pattern.

In [ ]:
# Purpose: Create and populate the company SQLite database
# Key Concept: We drop and recreate on each run so the notebook is idempotent;
#              a training database must be stable across episodes.

def create_company_database(db_path: Path) -> None:
    """
    Build the company training database from scratch.

    Parameters
    ----------
    db_path : Path
        File path for the SQLite database. Existing file is removed first.
    """
    # Step 1: Start fresh — a stable training environment must not carry
    #         over state from previous runs.
    if db_path.exists():
        db_path.unlink()

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Step 2: Define schema — three tables with explicit FK relationships
    cursor.executescript("""
        CREATE TABLE departments (
            id          INTEGER PRIMARY KEY,
            name        TEXT    NOT NULL,
            location    TEXT,
            budget      REAL
        );

        CREATE TABLE employees (
            id          INTEGER PRIMARY KEY,
            name        TEXT    NOT NULL,
            dept_id     INTEGER REFERENCES departments(id),
            salary      REAL,
            role        TEXT,
            is_active   INTEGER DEFAULT 1   -- 1 = active, 0 = former
        );

        CREATE TABLE projects (
            id          INTEGER PRIMARY KEY,
            name        TEXT    NOT NULL,
            dept_id     INTEGER REFERENCES departments(id),
            lead_id     INTEGER REFERENCES employees(id),
            budget      REAL,
            status      TEXT    -- 'active', 'completed', 'cancelled'
        );
    """)

    # Step 3: Insert departments
    cursor.executemany(
        "INSERT INTO departments (id, name, location, budget) VALUES (?, ?, ?, ?)",
        [
            (1, "Engineering",  "San Francisco", 1_500_000.0),
            (2, "Sales",        "New York",        800_000.0),
            (3, "Marketing",    "Chicago",         600_000.0),
            (4, "Operations",   "Austin",          400_000.0),
        ]
    )

    # Step 4: Insert employees — note Frank Lee (id=6) is inactive
    cursor.executemany(
        "INSERT INTO employees (id, name, dept_id, salary, role, is_active) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        [
            (1,  "Alice Chen",    1, 125_000.0, "Engineering Manager",  1),
            (2,  "Bob Martinez",  1,  98_000.0, "Senior Engineer",      1),
            (3,  "Carol Davis",   2,  75_000.0, "Sales Lead",           1),
            (4,  "Dan Wilson",    2,  68_000.0, "Account Executive",    1),
            (5,  "Eva Brown",     3,  82_000.0, "Marketing Director",   1),
            (6,  "Frank Lee",     3,  71_000.0, "Content Strategist",   0),
            (7,  "Grace Kim",     4,  65_000.0, "Operations Manager",   1),
            (8,  "Henry Patel",   1, 115_000.0, "Principal Engineer",   1),
            (9,  "Irene Nguyen",  1,  92_000.0, "Engineer",             1),
            (10, "James Taylor",  2,  78_000.0, "Sales Manager",        1),
        ]
    )

    # Step 5: Insert projects — Platform Migration is the most expensive active project
    cursor.executemany(
        "INSERT INTO projects (id, name, dept_id, lead_id, budget, status) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        [
            (1, "Platform Migration",  1, 1, 500_000.0, "active"),
            (2, "Mobile App v2",       1, 8, 350_000.0, "active"),
            (3, "Q4 Campaign",         3, 5, 120_000.0, "completed"),
            (4, "Sales Automation",    2, 3,  80_000.0, "active"),
            (5, "Cost Reduction",      4, 7,  45_000.0, "active"),
            (6, "Legacy Sunset",       1, 2, 200_000.0, "cancelled"),
        ]
    )

    conn.commit()
    conn.close()


create_company_database(DB_PATH)
print(f"Database created: {DB_PATH} ({DB_PATH.stat().st_size:,} bytes)")

### Verify the data with direct queries

Before building the MCP layer, confirm the database contains what you expect. These are raw `sqlite3` calls — not going through MCP tools. The MCP tools will wrap this same logic.

In [ ]:
# Purpose: Spot-check all three tables before wrapping them in MCP tools
# Key Concept: Always verify your data layer before adding abstraction on top.

def query_and_print(label: str, sql: str, db_path: Path = DB_PATH) -> list[dict]:
    """
    Run a SQL query and print results as a formatted table.

    Parameters
    ----------
    label : str
        Section header for display.
    sql : str
        SELECT statement to execute.
    db_path : Path
        SQLite database file.

    Returns
    -------
    list[dict]
        Query results as a list of row dictionaries.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.execute(sql)
    cols = [d[0] for d in cursor.description]
    rows = [dict(zip(cols, row)) for row in cursor.fetchall()]
    conn.close()

    print(f"\n--- {label} ---")
    for row in rows:
        print(" | ".join(f"{k}: {v}" for k, v in row.items()))
    return rows


# Departments overview
query_and_print(
    "Departments",
    "SELECT id, name, location, budget FROM departments ORDER BY budget DESC"
)

# Top 5 earners
query_and_print(
    "Top 5 Earners",
    "SELECT name, role, salary FROM employees WHERE is_active=1 ORDER BY salary DESC LIMIT 5"
)

# Active projects with leads
query_and_print(
    "Active Projects",
    """
    SELECT p.name AS project, e.name AS lead, p.budget
    FROM projects p
    JOIN employees e ON p.lead_id = e.id
    WHERE p.status = 'active'
    ORDER BY p.budget DESC
    """
)

## 3. Build the MCP Server

### How FastMCP works

FastMCP turns a decorated Python function into a fully MCP-compliant tool in three lines:

```python
mcp = fastmcp.FastMCP("server-name")

@mcp.tool()
def my_tool(param: str) -> str:
    """Docstring becomes the tool description the agent reads."""
    return do_work(param)
```

FastMCP reads the function signature to build a JSON Schema for inputs, reads the return type annotation to build an output schema, and uses the docstring as the tool description. The agent runtime (ART) reads these schemas at startup — that is tool discovery.

### Why three tools and not more?

```
list_tables()              → inventory: what exists?
describe_table(table_name) → schema: what columns?
run_query(sql)             → answer: what data?
```

This minimal set forces the agent to learn a strict order. Adding `count_rows` or `get_sample` creates shortcuts that bypass schema understanding — the agent learns to guess column names rather than inspect them.

In [ ]:
# Purpose: Define the complete MCP server with three database tools
# Key Concept: Each tool's docstring is the agent's only instruction about
#              when and how to call it — write docstrings for a language model,
#              not for a human developer.

# Server initialisation — name appears in agent logs and MCP metadata
mcp = fastmcp.FastMCP("company-database-server")


def _get_connection(db_path: Path) -> sqlite3.Connection:
    """
    Open a read-only SQLite connection using the URI filename format.

    Read-only mode (mode=ro) prevents agents or test cases from accidentally
    mutating the training database between episodes.

    Parameters
    ----------
    db_path : Path
        Absolute or relative path to the .db file.

    Returns
    -------
    sqlite3.Connection
        Open connection in read-only mode.
    """
    uri = f"file:{db_path}?mode=ro"
    return sqlite3.connect(uri, uri=True)


@mcp.tool()
def list_tables() -> list[str]:
    """
    Return the names of all user-created tables in the database.

    Call this first in any database task. It gives you the complete
    inventory of available data before inspecting individual schemas.

    Returns:
        List of table name strings, alphabetically sorted.

    Example:
        list_tables() -> ["departments", "employees", "projects"]
    """
    # Step 1: Open read-only connection — never modify the training database
    conn = _get_connection(DB_PATH)
    try:
        # Step 2: Query sqlite_master to get user table names
        #         sqlite_% prefix tables are internal SQLite metadata — exclude them
        cursor = conn.execute(
            "SELECT name FROM sqlite_master "
            "WHERE type='table' AND name NOT LIKE 'sqlite_%' "
            "ORDER BY name"
        )
        return [row[0] for row in cursor.fetchall()]
    finally:
        conn.close()


@mcp.tool()
def describe_table(table_name: str) -> list[dict[str, str]]:
    """
    Return column definitions for a table: name, type, and nullability.

    Call this before writing any query that references this table.
    Column names and types returned here are guaranteed accurate.
    Guessing column names without calling this tool causes query failures.

    Args:
        table_name: Exact table name as returned by list_tables().

    Returns:
        List of column dicts, each with keys:
          - "column":   column name
          - "type":     SQLite type (TEXT, INTEGER, REAL, BLOB, NUMERIC)
          - "nullable": "YES" or "NO"

    Raises:
        ValueError: If table_name does not exist in the database.

    Example:
        describe_table("employees") ->
        [
          {"column": "id",      "type": "INTEGER", "nullable": "NO"},
          {"column": "name",    "type": "TEXT",    "nullable": "NO"},
          {"column": "dept_id", "type": "INTEGER", "nullable": "YES"},
          {"column": "salary",  "type": "REAL",    "nullable": "YES"},
        ]
    """
    conn = _get_connection(DB_PATH)
    try:
        # PRAGMA table_info returns: cid, name, type, notnull, dflt_value, pk
        # SQLite returns empty results (not an error) for missing tables,
        # so we must check explicitly.
        cursor = conn.execute(f"PRAGMA table_info({table_name})")
        rows = cursor.fetchall()
        if not rows:
            raise ValueError(
                f"Table '{table_name}' not found. "
                f"Call list_tables() to see available tables."
            )
        return [
            {
                "column":   row[1],
                "type":     row[2] or "NUMERIC",  # empty type → NUMERIC affinity
                "nullable": "NO" if row[3] else "YES",
            }
            for row in rows
        ]
    finally:
        conn.close()


@mcp.tool()
def run_query(sql: str) -> list[dict[str, Any]]:
    """
    Execute a SQL SELECT query and return results as a list of row dicts.

    Only SELECT statements are permitted. Use list_tables() and
    describe_table() first to confirm table and column names before
    writing your query. Guessing column names causes OperationalError.

    Args:
        sql: A valid SQLite SELECT statement. Must begin with SELECT.

    Returns:
        List of row dicts mapping column name to value.
        Returns empty list when the query matches no rows.

    Raises:
        ValueError: If sql does not begin with SELECT.
        sqlite3.OperationalError: If sql contains a syntax error.

    Example:
        run_query("SELECT name, salary FROM employees LIMIT 3") ->
        [
          {"name": "Alice Chen",  "salary": 125000.0},
          {"name": "Bob Martinez","salary": 98000.0},
          {"name": "Henry Patel", "salary": 115000.0},
        ]
    """
    # Step 1: Enforce SELECT-only to prevent environment mutation
    #         strip() removes leading whitespace; upper() handles mixed case
    normalized = sql.strip().upper()
    if not normalized.startswith("SELECT"):
        raise ValueError(
            "Only SELECT queries are permitted. "
            f"Received statement starting with: {sql.strip()[:20]!r}"
        )

    # Step 2: Execute and convert rows to dicts using cursor.description for column names
    conn = _get_connection(DB_PATH)
    try:
        cursor = conn.execute(sql)
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()
        return [dict(zip(columns, row)) for row in rows]
    finally:
        conn.close()


print("MCP server defined with tools: list_tables, describe_table, run_query")
print(f"Server name: {mcp.name}")

## 4. Tool Discovery

When ART starts a training episode, the first thing it does is query the MCP server for its tool manifest. This is **tool discovery** — the server advertises what it can do, and the runtime translates those schemas into the agent's action space.

The cell below replicates that discovery step. `mcp.list_tools()` returns the same data the ART runtime reads at startup. The `inputSchema` field is a JSON Schema object — this is what constrains the agent's tool arguments during RL training.

In [ ]:
# Purpose: Inspect tool schemas the same way ART's MCPToolset does at startup
# Key Concept: The inputSchema is the agent's formal capability description —
#              each key in 'properties' is a named argument the agent can pass.

async def discover_tools(server: fastmcp.FastMCP) -> list[dict]:
    """
    Query the MCP server for its tool manifest.

    Parameters
    ----------
    server : fastmcp.FastMCP
        Initialised FastMCP server instance.

    Returns
    -------
    list[dict]
        List of tool metadata dicts with keys: name, description, inputSchema.
    """
    # list_tools() returns FunctionTool objects; to_mcp_tool() converts each
    # to the wire-format Tool object that ART reads.
    tools = await server.list_tools()
    manifest = []
    for tool in tools:
        mcp_tool = tool.to_mcp_tool()
        manifest.append({
            "name":        mcp_tool.name,
            "description": mcp_tool.description,
            "inputSchema": mcp_tool.inputSchema,
        })
    return manifest


tool_manifest = asyncio.run(discover_tools(mcp))

print("=" * 60)
print("TOOL MANIFEST (what ART reads at episode start)")
print("=" * 60)
for tool in tool_manifest:
    print(f"\nTool: {tool['name']}")
    print(f"  Description : {tool['description'][:80]}...")
    print(f"  inputSchema : {json.dumps(tool['inputSchema'], indent=4)}")

### What the agent sees

The manifest above is the agent's complete knowledge of its environment at the start of a training episode. Notice:

- `list_tables` has an empty `properties` dict — no arguments required
- `describe_table` requires exactly one string argument: `table_name`
- `run_query` requires exactly one string argument: `sql`

The JSON Schema `required` field enforces which arguments the agent must provide. ART uses this to validate the agent's output before calling the tool.

## 5. Agent Simulation

**Question:** *"Who leads the most expensive active project?"*

A trained agent answers this in three tool calls:

1. `list_tables()` — discover what tables exist  
2. `describe_table('projects')` + `describe_table('employees')` — learn column names  
3. `run_query(...)` — issue the JOIN query using correct column names  

The cells below simulate each step, print the tool call and result, and accumulate a **message history** — the same trajectory ART stores for reward computation.

**Try modifying the question** after you run this once. Change `'active'` to `'completed'` in the final query and observe how the answer changes.

In [ ]:
# Purpose: Build the trajectory data structure that ART uses during training
# Key Concept: Each message has a role (user/assistant/tool) and content;
#              ART computes rewards over the entire trajectory, not just the final answer.

def make_tool_call_message(tool_name: str, arguments: dict) -> dict:
    """
    Construct the assistant message that represents a tool call decision.

    Parameters
    ----------
    tool_name : str
        Name of the tool being called.
    arguments : dict
        Arguments to pass to the tool.

    Returns
    -------
    dict
        Message dict with role='assistant' and tool_call content.
    """
    return {
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "type":     "function",
            "function": {
                "name":      tool_name,
                "arguments": json.dumps(arguments)
            }
        }]
    }


def make_tool_result_message(tool_name: str, result: Any) -> dict:
    """
    Construct the tool message that carries a tool call result.

    Parameters
    ----------
    tool_name : str
        Name of the tool that was called.
    result : Any
        The value returned by the tool function.

    Returns
    -------
    dict
        Message dict with role='tool' and JSON-serialised content.
    """
    return {
        "role":    "tool",
        "name":    tool_name,
        "content": json.dumps(result)
    }


print("Message builder functions defined.")

### Running the trajectory

The function below runs all three steps in sequence and accumulates the message history. After running it once with `status = 'active'`, try changing the filter to `'completed'` in the final SQL string and rerun to see a different answer.

In [ ]:
# Purpose: Run the full three-step agent trajectory and display each turn
# Key Concept: The agent cannot answer the question without calling list_tables
#              and describe_table first — there is no shortcut.

async def run_agent_trajectory(question: str) -> list[dict]:
    """
    Simulate an agent solving a natural-language database question.

    The agent follows the canonical three-step pattern:
      Step 1: list_tables()            → discover available tables
      Step 2: describe_table() x N     → learn column names before writing SQL
      Step 3: run_query()              → answer the question

    Parameters
    ----------
    question : str
        Natural-language question about the database.

    Returns
    -------
    list[dict]
        Full message history (trajectory) for ART reward computation.
    """
    print(f"QUESTION: {question}")
    print("=" * 60)

    # Trajectory accumulates all messages for ART reward computation
    trajectory = [
        {"role": "user", "content": question}
    ]

    # ----------------------------------------------------------------
    # Step 1: list_tables() — discover the database inventory
    # ----------------------------------------------------------------
    print("\n[Step 1] Agent calls: list_tables()")
    trajectory.append(make_tool_call_message("list_tables", {}))

    result_1 = await mcp.call_tool("list_tables", {})
    tables = json.loads(result_1.content[0].text)
    trajectory.append(make_tool_result_message("list_tables", tables))

    print(f"  Result: {tables}")
    print("  Agent reasoning: I can see 'projects' and 'employees'. I need both.")

    # ----------------------------------------------------------------
    # Step 2a: describe_table('projects') — learn the projects schema
    # ----------------------------------------------------------------
    print("\n[Step 2a] Agent calls: describe_table('projects')")
    trajectory.append(make_tool_call_message("describe_table", {"table_name": "projects"}))

    result_2a = await mcp.call_tool("describe_table", {"table_name": "projects"})
    schema_projects = json.loads(result_2a.content[0].text)
    trajectory.append(make_tool_result_message("describe_table", schema_projects))

    col_names = [col["column"] for col in schema_projects]
    print(f"  Result columns: {col_names}")
    print("  Agent reasoning: 'status' filters active, 'budget' sorts by cost, 'lead_id' joins to employees.")

    # ----------------------------------------------------------------
    # Step 2b: describe_table('employees') — learn the employees schema
    # ----------------------------------------------------------------
    print("\n[Step 2b] Agent calls: describe_table('employees')")
    trajectory.append(make_tool_call_message("describe_table", {"table_name": "employees"}))

    result_2b = await mcp.call_tool("describe_table", {"table_name": "employees"})
    schema_employees = json.loads(result_2b.content[0].text)
    trajectory.append(make_tool_result_message("describe_table", schema_employees))

    col_names = [col["column"] for col in schema_employees]
    print(f"  Result columns: {col_names}")
    print("  Agent reasoning: 'name' is the employee name. JOIN key is employees.id = projects.lead_id.")

    # ----------------------------------------------------------------
    # Step 3: run_query() — execute the JOIN query
    # ----------------------------------------------------------------
    final_sql = """
    SELECT e.name AS lead_name, e.role, p.name AS project, p.budget
    FROM projects p
    JOIN employees e ON p.lead_id = e.id
    WHERE p.status = 'active'
    ORDER BY p.budget DESC
    LIMIT 1
    """.strip()

    print(f"\n[Step 3] Agent calls: run_query(...)")
    print(f"  SQL:\n    {final_sql.replace(chr(10), chr(10) + '    ')}")
    trajectory.append(make_tool_call_message("run_query", {"sql": final_sql}))

    result_3 = await mcp.call_tool("run_query", {"sql": final_sql})
    answer_rows = json.loads(result_3.content[0].text)
    trajectory.append(make_tool_result_message("run_query", answer_rows))

    print(f"  Result: {answer_rows}")

    # ----------------------------------------------------------------
    # Final answer: the agent synthesises the result into natural language
    # ----------------------------------------------------------------
    if answer_rows:
        row = answer_rows[0]
        answer = (
            f"{row['lead_name']} ({row['role']}) leads '{row['project']}' "
            f"with a budget of ${row['budget']:,.0f}."
        )
    else:
        answer = "No active projects found."

    trajectory.append({"role": "assistant", "content": answer})

    print(f"\nFINAL ANSWER: {answer}")
    print(f"\nTrajectory length: {len(trajectory)} messages")
    return trajectory


trajectory = asyncio.run(
    run_agent_trajectory("Who leads the most expensive active project?")
)

### Printing the full trajectory

Each message in the list below corresponds to one turn in the agent's conversation. ART reads this exact structure when computing the RULER reward at the end of the episode.

In [ ]:
# Purpose: Print the complete message trajectory that ART stores for reward computation
# Key Concept: ART scores the full trajectory — every tool call and result —
#              not just the final answer. Correct tool ordering is itself rewarded.

print("FULL MESSAGE TRAJECTORY")
print("=" * 60)
for i, message in enumerate(trajectory):
    role = message["role"].upper()
    if "tool_calls" in message:
        fn = message["tool_calls"][0]["function"]
        args = json.loads(fn["arguments"])
        args_str = ", ".join(f"{k}={v!r}" for k, v in args.items()) or "(no args)"
        print(f"[{i}] {role}: CALL {fn['name']}({args_str})")
    elif role == "TOOL":
        content_preview = message["content"][:80]
        print(f"[{i}] {role} ({message['name']}): {content_preview}...")
    else:
        content = str(message.get("content", ""))[:100]
        print(f"[{i}] {role}: {content}")

### What ART does with this trajectory

ART applies the RULER reward function to the trajectory above:

- **Tool order reward**: `list_tables` called before `describe_table`? +0.2
- **Schema inspection reward**: `describe_table` called for every referenced table? +0.3  
- **Query correctness reward**: Does the SQL execute without error? +0.2  
- **Answer accuracy reward**: Does the answer match the gold label? +0.3  

An agent that skips Steps 1–2 and guesses column names loses the first 0.5 of reward, even if the final SQL accidentally works. That gradient signal is what drives the schema-first pattern.

## 6. Exercise: Build a Bookstore MCP Server

**Task:** Create a complete MCP server for a bookstore database. The server must have:

1. A SQLite database with at least three tables: `books`, `authors`, and `orders`
2. Three tools: `list_tables`, `describe_table`, and `run_query` (adapted for the bookstore)
3. A sample query that answers: *"Which author has the highest total order value?"*

**Minimum schema requirements:**
- `books`: at least `id`, `title`, `author_id`, `price`, `genre`
- `authors`: at least `id`, `name`, `country`
- `orders`: at least `id`, `book_id`, `quantity`, `total_price`, `status`

**Expected output from the sample query:** A single row with at least `author_name` and `total_revenue` columns.

**Hints:**
<details>
<summary>Hint 1 — Database setup</summary>
Copy the `create_company_database` pattern. Define schema with `executescript`, then insert rows with `executemany`. Include at least one completed order per author to make the aggregation meaningful.
</details>

<details>
<summary>Hint 2 — Tool reuse</summary>
The three tool functions are almost identical to the company server. The only difference is the module-level `DB_PATH` variable. Create a new `FastMCP` instance (`bookstore_mcp`) and a new database file (`bookstore.db`), then copy the tool definitions — they close over whichever `DB_PATH` is in scope when the function is defined.
</details>

<details>
<summary>Hint 3 — Sample query structure</summary>
The query needs to JOIN `orders` to `books` to `authors`, GROUP BY `author_id`, SUM `total_price`, and ORDER BY the sum descending. Filter on `status = 'completed'` to count only fulfilled orders.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Part A: Create the bookstore database

BOOKSTORE_DB_PATH = Path("bookstore.db")

def create_bookstore_database(db_path: Path) -> None:
    """
    Build the bookstore SQLite database.

    Parameters
    ----------
    db_path : Path
        File path for the SQLite database.
    """
    # TODO: Implement this function
    # - Drop and recreate the database (same pattern as create_company_database)
    # - Create tables: books, authors, orders
    # - Insert at least 4 authors, 8 books, and 10 orders
    pass


# Uncomment when your implementation is ready:
# create_bookstore_database(BOOKSTORE_DB_PATH)

bookstore_db_created = False  # Replace with True after implementation

### Part B: Define the MCP server tools

Create a new `FastMCP` instance called `bookstore_mcp` and attach the three tools to it. The tools are structurally identical to the company server — the only change is the database path.

In [ ]:
# YOUR CODE HERE
# ---------------
# Part B: Create the bookstore MCP server
# Create a new FastMCP instance and define the three tools.
# The tools should read from BOOKSTORE_DB_PATH, not DB_PATH.

bookstore_mcp = None  # Replace with: fastmcp.FastMCP("bookstore-server")

# TODO: Define list_tables, describe_table, run_query tools on bookstore_mcp
# These should be identical in structure to the company server tools,
# but use BOOKSTORE_DB_PATH instead of DB_PATH.

bookstore_tools_defined = False  # Replace with True after implementation

### Part C: Write the answer query

Write a `SELECT` statement that answers *"Which author has the highest total order value?"*. Your result must include columns named `author_name` and `total_revenue`.

In [ ]:
# YOUR CODE HERE
# ---------------
# Part C: Write the sample query
# Answer: "Which author has the highest total order value?"

SAMPLE_QUERY = None  # Replace with your SQL string

# The query should return rows with at least:
#   - author_name : str
#   - total_revenue : float
# Ordered by total_revenue DESC, LIMIT 1

### Auto-graded tests

Run this cell after completing Parts A, B, and C. Each test checks one specific requirement and prints a diagnostic message if something is wrong. Fix the indicated issue, re-run the three exercise cells above, then re-run this cell.

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------

async def _run_bookstore_tests():
    """Run all bookstore exercise tests."""
    errors = []

    # Test 1: Database was created
    assert bookstore_db_created, (
        "bookstore_db_created is False. "
        "Call create_bookstore_database(BOOKSTORE_DB_PATH) and set bookstore_db_created = True."
    )
    assert BOOKSTORE_DB_PATH.exists(), (
        f"{BOOKSTORE_DB_PATH} does not exist. "
        "Make sure create_bookstore_database() creates the file."
    )

    # Test 2: Database has the required tables
    conn = sqlite3.connect(BOOKSTORE_DB_PATH)
    cursor = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    )
    actual_tables = {row[0] for row in cursor.fetchall()}
    conn.close()

    required_tables = {"books", "authors", "orders"}
    missing = required_tables - actual_tables
    assert not missing, (
        f"Missing required tables: {missing}. "
        f"Found: {actual_tables}. Create all three tables in create_bookstore_database()."
    )

    # Test 3: Each table has data
    conn = sqlite3.connect(BOOKSTORE_DB_PATH)
    for table in required_tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        assert count > 0, (
            f"Table '{table}' is empty. Insert at least one row into every table."
        )
    conn.close()

    # Test 4: Bookstore MCP server is defined
    assert bookstore_tools_defined, (
        "bookstore_tools_defined is False. "
        "Define the three tools on bookstore_mcp and set bookstore_tools_defined = True."
    )
    assert bookstore_mcp is not None, (
        "bookstore_mcp is None. Create a FastMCP instance: bookstore_mcp = fastmcp.FastMCP('bookstore-server')."
    )

    bookstore_tool_list = await bookstore_mcp.list_tools()
    bookstore_tool_names = {t.name for t in bookstore_tool_list}
    required_tool_names = {"list_tables", "describe_table", "run_query"}
    missing_tools = required_tool_names - bookstore_tool_names
    assert not missing_tools, (
        f"bookstore_mcp is missing tools: {missing_tools}. "
        f"Found: {bookstore_tool_names}. Define all three with @bookstore_mcp.tool()."
    )

    # Test 5: list_tables returns the required tables
    result = await bookstore_mcp.call_tool("list_tables", {})
    returned_tables = set(json.loads(result.content[0].text))
    assert required_tables.issubset(returned_tables), (
        f"list_tables() returned {returned_tables}, expected at least {required_tables}."
    )

    # Test 6: describe_table returns column info
    result = await bookstore_mcp.call_tool("describe_table", {"table_name": "authors"})
    columns = json.loads(result.content[0].text)
    assert isinstance(columns, list) and len(columns) > 0, (
        "describe_table('authors') returned an empty list. "
        "Make sure the authors table has columns."
    )
    col_keys = set(columns[0].keys())
    assert {"column", "type", "nullable"}.issubset(col_keys), (
        f"Each column dict must have keys 'column', 'type', 'nullable'. Got: {col_keys}"
    )

    # Test 7: Sample query works and returns expected shape
    assert SAMPLE_QUERY is not None, (
        "SAMPLE_QUERY is None. Set it to a SQL string that answers the author revenue question."
    )
    result = await bookstore_mcp.call_tool("run_query", {"sql": SAMPLE_QUERY})
    rows = json.loads(result.content[0].text)
    assert isinstance(rows, list) and len(rows) >= 1, (
        "SAMPLE_QUERY returned no rows. Check that orders table has 'completed' status rows."
    )
    first_row = rows[0]
    assert "author_name" in first_row, (
        f"Expected 'author_name' in result row. Got keys: {list(first_row.keys())}. "
        "Use 'AS author_name' in your SELECT."
    )
    assert "total_revenue" in first_row, (
        f"Expected 'total_revenue' in result row. Got keys: {list(first_row.keys())}. "
        "Use 'AS total_revenue' in your SELECT."
    )
    assert isinstance(first_row["total_revenue"], (int, float)), (
        f"'total_revenue' must be numeric, got {type(first_row['total_revenue'])}."
    )

    # Test 8: run_query correctly rejects non-SELECT statements
    try:
        await bookstore_mcp.call_tool("run_query", {"sql": "DELETE FROM books WHERE id=1"})
        assert False, (
            "run_query did not raise an error for a DELETE statement. "
            "Add a check: if not sql.strip().upper().startswith('SELECT'): raise ValueError(...)"
        )
    except Exception as e:
        if "Only SELECT" not in str(e) and "SELECT" not in str(e):
            raise AssertionError(
                f"run_query raised the wrong error for DELETE: {e}. "
                "Raise a ValueError with a message mentioning SELECT."
            )

    print("Exercise 6 passed — bookstore MCP server works correctly.")
    print(f"  Tables   : {sorted(returned_tables)}")
    print(f"  Top author: {first_row['author_name']} — ${first_row['total_revenue']:,.2f}")


try:
    asyncio.run(_run_bookstore_tests())
except AssertionError as _exc:
    print(f"Exercise not yet complete: {_exc}")
    print("Implement Parts A, B, and C above, then re-run this cell.")

## 7. Summary

**Key takeaways:**

1. **FastMCP turns decorated Python functions into MCP tools.** The function signature becomes the input JSON Schema; the docstring becomes the agent's only instruction about the tool.

2. **Three tools are enough for any relational database.** `list_tables` → `describe_table` → `run_query` enforces the schema-first exploration pattern that generalises across databases.

3. **Read-only connections prevent training instability.** Opening SQLite with `file:path?mode=ro` ensures no agent or test run can modify the training data.

4. **Tool discovery is the first thing ART does.** `mcp.list_tools()` returns the manifest that defines the agent's action space for the entire episode.

5. **The full trajectory — not just the final answer — is the training signal.** ART rewards correct tool ordering and schema inspection steps, not just the SQL result.

**What's next:**
- **Module 05 (Training Loop):** Connect this MCP server to ART and run a full GRPO training loop
- **Module 06 (Text-to-SQL Agent):** Train a production agent against this server with 500+ question scenarios
- **Module 07 (Production):** Deploy the MCP server behind authentication for live database access

**Additional resources:**
- [FastMCP documentation](https://gofastmcp.com/)
- [Model Context Protocol specification](https://modelcontextprotocol.io/)
- Guide 02 in this module: `guides/02_fastmcp_server_guide.md` — complete server reference including transport options and testing patterns